# 03 - Class steering with Gaussian/PCA denoisers

Notebook `02` showed that a velocity model can also be read as a denoiser. We
now ask whether a simpler denoiser, built only from a distribution's mean and
principal directions, can supply useful class information while the sample is
still dominated by noise. We first test Gaussian/PCA posterior means on
held-out MNIST images, then compare a target-class denoiser with a full-data
denoiser at the same 2D state. Their difference,

$$\Delta D = D_{\text{target}} - D_{\text{full}},$$

is a coarse distribution-level correction. It does not pull the sample toward
a chosen image, centroid, or coordinate.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *
from notebook_views import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print("device:", device)


In [ ]:
model, losses, _ = load_or_train_model(device=device)
model.eval()
model.requires_grad_(False)

set_seed(SEED + 301)
steering_data, steering_labels = sample_labeled_mixture(30000, device=device)
steering_stats = estimate_gaussian_stats(steering_data, steering_labels)
set_seed(SEED + 302)
evaluation_data, evaluation_labels = sample_labeled_mixture(30000, device=device)
evaluation_stats = estimate_gaussian_stats(evaluation_data, evaluation_labels)
TARGET_CLASS = 2
set_seed(SEED + 303)
target_reference = sample_class(1800, TARGET_CLASS, device=device)
print("target class:", CLASS_NAMES[TARGET_CLASS])


## 1. Why a Gaussian denoiser makes sense at high noise

Write the base point as $x_{noise}=s\epsilon$ with
$\epsilon\sim\mathcal{N}(0,I)$ and `s=1.8`, then divide the path by `t`:

$$\frac{x_t}{t}=x_{data}+s\frac{1-t}{t}\epsilon.$$

The right side is clean data plus Gaussian noise with effective noise level
`sigma_eff=s(1-t)/t`. For a Gaussian data model, the posterior mean is

$$D(y,\sigma)=\mu+C(C+\sigma^2I)^{-1}(y-\mu).$$

If $C=U\Lambda U^\top$, each principal-component coordinate is multiplied by
$\lambda_i/(\lambda_i+\sigma^2)$. Large-variance directions survive; uncertain
small-variance directions shrink toward the mean. This is why PCA is a useful
way to compute the same rule for images.

The factor `s` matters because our base distribution is not unit variance.
At high noise the denoiser trusts coarse mean/covariance statistics; at low noise
it stays close to the observed point.

The full-data Gaussian is deliberately crude because the complete dataset is a
mixture. That limitation is useful: subtracting the full denoiser from the
target-class denoiser isolates a coarse class correction.

[Li, Dai, and Qu (NeurIPS 2024)](https://arxiv.org/abs/2410.24060) provide
broader evidence that learned diffusion denoisers can approach empirical
Gaussian denoisers in a generalization regime. The MNIST calculation below is
our teaching example; Notebook `06` supplies the pretrained image-generator
experiment for this series.


In [ ]:
def gaussian_posterior_mean(y, sigma, mean, covariance):
    """E[x_clean | y] when x_clean is Gaussian and y = x_clean + noise."""
    sigma = torch.as_tensor(sigma, device=y.device).reshape(-1)
    if sigma.numel() == 1:
        sigma = sigma.expand(y.shape[0])
    identity = torch.eye(y.shape[1], device=y.device).expand(y.shape[0], -1, -1)
    covariance = covariance.expand(y.shape[0], -1, -1)
    system = covariance + sigma[:, None, None].square() * identity
    solved = torch.linalg.solve(system, (y - mean).unsqueeze(-1))
    return mean + (covariance @ solved).squeeze(-1)

def pca_posterior_mean(y, sigma, stats):
    centered = y - stats.mean
    coordinates = centered @ stats.components
    sigma_squared = torch.as_tensor(sigma, device=y.device).reshape(-1, 1).square()
    variances = stats.variances[None, :]
    shrinkage = variances / (variances + sigma_squared)
    return stats.mean + (coordinates * shrinkage) @ stats.components.T


## 2. Does a covariance model retain useful image structure?

MNIST makes all 784 pixel coordinates visible without needing a large network.
We fit low-rank Gaussian models on the training split only: one to all digits
and one to digit 3. The leading covariance directions are `eigendigits`.
On held-out test images, we add known Gaussian noise and compare the noisy input
with both posterior-mean denoisers.


In [ ]:
MNIST_TARGET = 3
MNIST_RANK = 64
MNIST_SIGMA = 0.45

mnist_train_images, mnist_train_labels = load_mnist_split(train=True, device=device)
mnist_test_images, mnist_test_labels = load_mnist_split(train=False, device=device)

full_fit_indexes = torch.linspace(
    0, mnist_train_images.shape[0] - 1, 12000, device=device
).round().long()
class_fit_pool = torch.where(mnist_train_labels == MNIST_TARGET)[0]
class_fit_indexes = class_fit_pool[: min(6000, class_fit_pool.numel())]

set_seed(SEED + 330)
mnist_full_gaussian = fit_low_rank_gaussian(
    mnist_train_images[full_fit_indexes], rank=MNIST_RANK
)
set_seed(SEED + 331)
mnist_class_gaussian = fit_low_rank_gaussian(
    mnist_train_images[class_fit_indexes], rank=MNIST_RANK
)

print("MNIST training images used for full Gaussian:", full_fit_indexes.numel())
print("MNIST digit-3 training images used:", class_fit_indexes.numel())
print("PCA rank:", MNIST_RANK)


In [ ]:
plot_pca_components(mnist_full_gaussian, mnist_class_gaussian)


In [ ]:
held_out_pool = torch.where(mnist_test_labels == MNIST_TARGET)[0]
held_out_indexes = held_out_pool[:256]
mnist_clean = mnist_test_images[held_out_indexes]
set_seed(SEED + 332)
mnist_noisy = mnist_clean + MNIST_SIGMA * torch.randn_like(mnist_clean)
mnist_full_denoised = pca_posterior_mean(mnist_noisy, MNIST_SIGMA, mnist_full_gaussian)
mnist_class_denoised = pca_posterior_mean(mnist_noisy, MNIST_SIGMA, mnist_class_gaussian)

def image_mse(images):
    return float(torch.mean((images - mnist_clean).square()).cpu())

mnist_mse = {
    "noisy input": image_mse(mnist_noisy),
    "all-digit Gaussian": image_mse(mnist_full_denoised),
    "digit-3 Gaussian": image_mse(mnist_class_denoised),
}
mnist_metrics = pd.DataFrame({
    "method": list(mnist_mse),
    "held-out MSE": list(mnist_mse.values()),
    "PSNR (dB)": [10 * np.log10(1.0 / value) for value in mnist_mse.values()],
})
display(mnist_metrics.round(4))

plot_image_rows([
    ("clean test image", mnist_clean),
    (f"noisy, sigma={MNIST_SIGMA}", mnist_noisy),
    ("all-digit Gaussian", mnist_full_denoised),
    ("digit-3 Gaussian", mnist_class_denoised),
], title="Gaussian/PCA posterior means on held-out digit-3 images")


## 3. How does denoising become a steering signal?

The image example shows what one Gaussian posterior mean preserves and loses.
For steering, we compare two such estimates at the same noisy state. Their
difference asks for class-level structure that the target model favors more
than the full-data model. The gate below limits that correction to the
high-noise part of the trajectory.


In [ ]:
grid_1d = torch.linspace(-4.5, 4.5, 19, device=device)
gx, gy = torch.meshgrid(grid_1d, grid_1d, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
t_value = 0.18
t = torch.full((grid.shape[0], 1), t_value, device=device)
y = grid / t
sigma = BASE_STD * (1 - t_value) / t_value

D_target = gaussian_posterior_mean(
    y, sigma, steering_stats.means[TARGET_CLASS], steering_stats.covariances[TARGET_CLASS]
)
D_full = gaussian_posterior_mean(
    y, sigma, steering_stats.full_mean, steering_stats.full_covariance
)
denoiser_delta = D_target - D_full

plot_noise_alignment_field(
    grid, denoiser_delta, steering_data, steering_labels,
    target_class=TARGET_CLASS, t_value=t_value, start=0.04, end=0.35,
)


## 4. Does the correction improve paired samples?

All methods below use the same initial noise, Heun solver, step count, and target
reference. The only change is the noise-alignment strength. The correction is
active for `t in [0.04, 0.35]`, the high-noise part of our noise-to-data path.


In [ ]:
def noise_aligned_velocity(strength):
    def velocity(t, x):
        with torch.no_grad():
            base = model(t, x)
            t_column = t[:, None].clamp_min(0.04)
            y = x / t_column
            sigma = BASE_STD * (1 - t) / t.clamp_min(0.04)
            target_estimate = gaussian_posterior_mean(
                y, sigma, steering_stats.means[TARGET_CLASS],
                steering_stats.covariances[TARGET_CLASS],
            )
            full_estimate = gaussian_posterior_mean(
                y, sigma, steering_stats.full_mean, steering_stats.full_covariance,
            )
            gate = window_gate(t, 0.04, 0.35)[:, None]
            remaining = (1 - t_column).clamp_min(0.04)
            return base + strength * gate * (target_estimate - full_estimate) / remaining
    return velocity


In [ ]:
set_seed(SEED + 30)
x0 = sample_base(1800, device=device)
strengths = [0.0, 0.4, 0.8, 1.2, 1.6]
methods = {"baseline": base_velocity(model)}
methods.update({f"noise alignment {s:.1f}": noise_aligned_velocity(s) for s in strengths})

rows, paths = compare_steering_methods(
    methods, x0, target_reference, evaluation_stats,
    target_class=TARGET_CLASS, steps=64,
)
noise_alignment_table = pd.DataFrame(rows)
noise_alignment_table["strength"] = [
    np.nan if name == "baseline" else float(name.rsplit(" ", 1)[-1])
    for name in noise_alignment_table["method"]
]
display(noise_alignment_table.round(4))


In [ ]:
plot_noise_alignment_comparison(
    paths, noise_alignment_table, target_reference, evaluation_stats, strength=1.6,
)


In [ ]:
selected = noise_alignment_table.set_index("method").loc[
    ["baseline", "noise alignment 1.6"],
    ["target_rate", "target_wasserstein", "mean_error", "diversity_ratio"],
]
display(selected.round(4))


## Reading the paired result

Read the endpoints and metrics together. A higher target-class rate indicates
stronger control, a lower Wasserstein distance indicates better agreement with
target examples, and the diversity ratio shows whether that gain came from
collapse or excessive spread. The model remains unconditional and unchanged;
only the target-class and full-data summary statistics are prepared offline.
In 2D both principal directions are retained, while an image implementation
uses a low-rank PCA covariance model.

**Questions**

1. Why should this coarse correction be most useful at high noise?
2. Which strength best balances target rate and target Wasserstein distance in your run?
3. What information is lost when a multimodal full dataset is approximated by one Gaussian?

In this toy setting, the class-minus-full Gaussian correction produces partial
coarse steering by a frozen unconditional model; it does not recover a full
class-conditional distribution. Because means and covariances can express only
limited structure, Notebook `04` replaces those fixed summaries with the
gradient of a differentiable class objective.
